## Download Boston Airbnb Dataset

In [30]:
# !wget https://data.insideairbnb.com/united-states/ma/boston/2025-12-27/data/listings.csv.gz
# !gunzip listings.csv.gz

In [31]:
!wget https://data.insideairbnb.com/united-states/ma/boston/2025-09-23/data/listings.csv.gz
!gunzip listings.csv.gz

--2026-04-22 18:57:44--  https://data.insideairbnb.com/united-states/ma/boston/2025-09-23/data/listings.csv.gz
Resolving data.insideairbnb.com (data.insideairbnb.com)... 13.35.37.68, 13.35.37.7, 13.35.37.10, ...
Connecting to data.insideairbnb.com (data.insideairbnb.com)|13.35.37.68|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2263228 (2.2M) [application/x-gzip]
Saving to: ‘listings.csv.gz.3’

listings.csv.gz.3   100%[===================>]   2.16M  --.-KB/s    in 0.04s   

2026-04-22 18:57:44 (53.6 MB/s) - ‘listings.csv.gz.3’ saved [2263228/2263228]

gzip: listings.csv already exists; do you wish to overwrite (y or n)? y


## Install required libraries

In [32]:
!pip install scikit-learn scipy pandas numpy joblib

## Import libraries

In [33]:
import ast
import re
import warnings
from dataclasses import dataclass
from typing import List, Optional, Dict, Any

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## Load Dataset

In [34]:
df_raw = pd.read_csv("listings.csv")
print(df_raw.shape)
df_raw.head()

(4419, 79)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,3781,https://www.airbnb.com/rooms/3781,20250923202714,2025-09-23,city scrape,HARBORSIDE-Walk to subway,Fully separate apartment in a two apartment bu...,"Mostly quiet ( no loud music, no crowed sidewa...",https://a0.muscache.com/pictures/24670/b2de044...,4804,https://www.airbnb.com/users/show/4804,Frank,2008-12-03,"Massachusetts, United States",My wife and I and grown children frequently oc...,within a day,100%,19%,t,https://a0.muscache.com/im/users/4804/profile_...,https://a0.muscache.com/im/users/4804/profile_...,East Boston,3.0,6.0,"['email', 'phone']",t,f,Neighborhood highlights,East Boston,NaN,42.36413,-71.02991,Entire rental unit,Entire home/apt,2,1.0,1 bath,1.0,1.0,"[""Heating"", ""Dishwasher"", ""Smoke alarm"", ""Iron...",$125.00,29,1125,29.0,29.0,1125.0,1125.0,29.0,1125.0,NaN,t,3,21,51,326,2025-09-23,26,0,0,61,1,0,0.0,2015-07-10,2024-08-09,4.96,5.00,4.96,5.00,4.96,4.85,4.88,NaN,f,1,1,0,0,0.21
1,5506,https://www.airbnb.com/rooms/5506,20250923202714,2025-09-24,city scrape,** Fort Hill Inn Private! Minutes to center!**,**THE BEST Value in BOSTON!!*** PRIVATE GUEST ...,"Peaceful, Architecturally interesting, histori...",https://a0.muscache.com/pictures/miso/Hosting-...,8229,https://www.airbnb.com/users/show/8229,Terry,2009-02-19,"Boston, MA","Relaxed, Easy going, Accommodating.",within an hour,100%,100%,t,https://a0.muscache.com/im/users/8229/profile_...,https://a0.muscache.com/im/users/8229/profile_...,Roxbury,12.0,15.0,"['email', 'phone']",t,t,Neighborhood highlights,Roxbury,NaN,42.32844,-71.09581,Entire guest suite,Entire home/apt,2,1.0,1 bath,1.0,1.0,"[""Heating"", ""Smoke alarm"", ""Private entrance"",...",$129.00,3,90,1.0,3.0,1125.0,1125.0,3.0,1125.0,NaN,t,3,28,58,67,2025-09-24,138,9,0,67,10,54,6966.0,2009-03-21,2025-07-28,4.82,4.89,4.91,4.95,4.90,4.58,4.77,STR-490093,f,11,11,0,0,0.69
2,6695,https://www.airbnb.com/rooms/6695,20250923202714,2025-09-24,city scrape,"Fort Hill Inn *Sunny* 1 bedroom, condo duplex","Comfortable, Fully Equipped private apartment...","Peaceful, Architecturally interesting, histori...",https://a0.muscache.com/pictures/38ac4797-e7a4...,8229,https://www.airbnb.com/users/show/8229,Terry,2009-02-19,"Boston, MA","Relaxed, Easy going, Accommodating.",within an hour,100%,100%,t,https://a0.muscache.com/im/users/8229/profile_...,https://a0.muscache.com/im/users/8229/profile_...,Roxbury,12.0,15.0,"['email', 'phone']",t,t,Neighborhood highlights,Roxbury,NaN,42.32802,-71.09387,Entire condo,Entire home/apt,4,1.0,1 bath,0.0,2.0,"[""Heating"", ""Dishwasher"", ""Smoke alarm"", ""Iron...",$168.00,3,730,3.0,3.0,730.0,730.

## Check available columns

In [35]:
df_raw.columns.tolist()

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_name',
 'host_since',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'calendar_updated',
 'has_availability',
 'availability_30

## Helper functions

In [36]:
def clean_price(value):
    if pd.isna(value):
        return np.nan
    value = str(value)
    value = re.sub(r"[$,]", "", value)
    try:
        return float(value)
    except:
        return np.nan


def parse_amenities(amenities):
    if pd.isna(amenities):
        return ""

    text = str(amenities)

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            items = [str(x).lower().strip() for x in parsed]
        else:
            items = [text.lower()]
    except:
        text = text.replace("{", "").replace("}", "").replace('"', "")
        items = [x.strip().lower() for x in text.split(",") if x.strip()]

    cleaned = []
    for item in items:
        item = re.sub(r"[^a-zA-Z0-9 ]", " ", item)
        item = re.sub(r"\s+", " ", item).strip()
        if item:
            cleaned.append(item.replace(" ", "_"))

    return " ".join(cleaned)


def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

Select useful columns and clean data

In [37]:
desired_columns = [
    "id",
    "name",
    "description",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms_text",
    "bedrooms",
    "beds",
    "amenities",
    "price",
    "minimum_nights",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "instant_bookable"
]

existing_columns = [col for col in desired_columns if col in df_raw.columns]
df = df_raw[existing_columns].copy()

if "price" in df.columns:
    df["price"] = df["price"].apply(clean_price)

if "amenities" in df.columns:
    df["amenities_clean"] = df["amenities"].apply(parse_amenities)
else:
    df["amenities_clean"] = ""

if "description" in df.columns:
    df["description_clean"] = df["description"].apply(clean_text)
else:
    df["description_clean"] = ""

for col in ["instant_bookable", "room_type", "property_type", "neighbourhood_cleansed"]:
    if col in df.columns:
        df[col] = df[col].astype(str)

numeric_cols = [
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "minimum_nights",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "latitude",
    "longitude"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

if "price" in df.columns:
    upper_price = df["price"].quantile(0.99)
    df = df[(df["price"].isna()) | (df["price"] <= upper_price)].reset_index(drop=True)

print(df.shape)
df.head()

(4397, 23)


,id,name,description,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,availability_365,number_of_reviews,review_scores_rating,review_scores_cleanliness,review_scores_location,instant_bookable,amenities_clean,description_clean
0,3781,HARBORSIDE-Walk to subway,Fully separate apartment in a two apartment bu...,East Boston,42.36413,-71.02991,Entire rental unit,Entire home/apt,2,1 bath,1.0,1.0,"[""Heating"", ""Dishwasher"", ""Smoke alarm"", ""Iron...",125.0,29,326,26,4.96,4.96,4.85,f,heating dishwasher smoke_alarm iron carbon_mon...,fully separate apartment in a two apartment bu...
1,5506,** Fort Hill Inn Private! Minutes to center!**,**THE BEST Value in BOSTON!!*** PRIVATE GUEST ...,Roxbury,42.32844,-71.09581,Entire guest suite,Entire home/apt,2,1 bath,1.0,1.0,"[""Heating"", ""Smoke alarm"", ""Private entrance"",...",129.0,3,67,138,4.82,4.91,4.58,f,heating smoke_alarm private_entrance iron carb...,the best value in boston private guest room wi...
2,6695,"Fort Hill Inn *Sunny* 1 bedroom, condo duplex","Comfortable, Fully Equipped private apartment...",Roxbury,42.32802,-71.09387,Entire condo,Entire home/apt,4,1 bath,0.0,2.0,"[""Heating"", ""Dishwasher"", ""Smoke alarm"", ""Iron...",168.0,3,56,141,4.81,4.86,4.54,f,heating dishwasher smoke_alarm iron carbon_mon...,comfortable fully equipped private apartment o...
3,8789,Curved Glass Studio/1bd facing Park,This unit is for sale. There will need to be o...,Beacon Hill,42.35867,-71.06307,Entire rental unit,Entire home/apt,2,1 bath,1.0,2.0,"[""Heating"", ""Smoke alarm"", ""Elevator"", ""Iron"",...",140.0,91,277,29,4.69,4.55,4.97,f,heating smoke_alarm elevator iron carbon_monox...,this unit is for sale there will need to be oc...
4,10811,Bostons Best Rentals -Studio Prestigious Back Bay,Bostons Best Rentals offers a Stunning Back Ba...,Back Bay,42.35173,-71.08685,Entire rental unit,Entire home/apt,3,1 bath,0.0,1.0,"[""Heating"", ""Smoke alarm"", ""Iron"", ""Carbon mon...",166.0,91,218,9,4.33,4.67,5.00,f,heating smoke_alarm iron carbon_monoxide_alarm...,bostons best rentals offers a stunning back ba...


Missing values check

In [38]:
df.isnull().sum().sort_values(ascending=False).head(20)

,0
review_scores_location,962
review_scores_cleanliness,960
review_scores_rating,960
price,913
beds,869
bedrooms,305
description,47
bathrooms_text,15
longitude,0
property_type,0


In [39]:
df["bathrooms_text"].value_counts()

,count
bathrooms_text,
1 bath,2230
2 baths,581
1 shared bath,565
1 private bath,324
2 shared baths,166
1.5 baths,152
1.5 shared baths,81
2.5 baths,79
0 shared baths,67


In [40]:
import numpy as np
import pandas as pd

# Fill missing first
df["bathrooms_text"] = df["bathrooms_text"].fillna("Unknown")

# Standardize text
df["bathrooms_text_clean"] = (
    df["bathrooms_text"]
    .str.lower()
    .str.strip()
)

# Extract numeric bathroom count like 1, 1.5, 2, 2.5
df["bathrooms_count"] = df["bathrooms_text_clean"].str.extract(r"(\d+\.?\d*)")[0]

# Convert to numeric
df["bathrooms_count"] = pd.to_numeric(df["bathrooms_count"], errors="coerce")

# Handle half-bath cases where no leading number exists
df.loc[df["bathrooms_text_clean"].str.contains("half-bath", na=False), "bathrooms_count"] = 0.5

# Fill remaining missing bathroom counts with median
df["bathrooms_count"] = df["bathrooms_count"].fillna(df["bathrooms_count"].median())

# Shared/private flags
df["is_shared_bath"] = df["bathrooms_text_clean"].str.contains("shared", na=False).astype(int)
df["is_private_bath"] = df["bathrooms_text_clean"].str.contains("private", na=False).astype(int)
df["is_half_bath"] = df["bathrooms_text_clean"].str.contains("half-bath", na=False).astype(int)

# Quick check
df[["bathrooms_text", "bathrooms_count", "is_shared_bath", "is_private_bath", "is_half_bath"]].head(15)

,bathrooms_text,bathrooms_count,is_shared_bath,is_private_bath,is_half_bath
0,1 bath,1.0,0,0,0
1,1 bath,1.0,0,0,0
2,1 bath,1.0,0,0,0
3,1 bath,1.0,0,0,0
4,1 bath,1.0,0,0,0
5,1 bath,1.0,0,0,0
6,1 bath,1.0,0,0,0
7,1 bath,1.0,0,0,0
8,1 bath,1.0,0,0,0
9,1.5 baths,1.5,0,0,0


In [41]:
df.isnull().sum().sort_values(ascending=False).head(20)

,0
review_scores_location,962
review_scores_rating,960
review_scores_cleanliness,960
price,913
beds,869
bedrooms,305
description,47
longitude,0
neighbourhood_cleansed,0
name,0


## Basic Exploration

In [42]:
print("Unique neighborhoods:", df["neighbourhood_cleansed"].nunique())
print("Unique room types:", df["room_type"].nunique())
print("Unique property types:", df["property_type"].nunique())

df[["price", "review_scores_rating", "accommodates", "bedrooms"]].describe()

Unique neighborhoods: 25
Unique room types: 4
Unique property types: 32


,price,review_scores_rating,accommodates,bedrooms
count,3484.000000,3437.000000,4397.000000,4092.000000
mean,459.113662,4.726593,3.371617,1.532502
std,2871.736430,0.408737,2.469964,1.179209
min,26.000000,1.000000,1.000000,0.000000
25%,118.000000,4.670000,2.000000,1.000000
50%,204.000000,4.830000,2.000000,1.000000
75%,312.000000,4.970000,4.000000,2.000000
max,40000.000000,5.000000,16.000000,15.000000


## Define features for recommendation

In [43]:
numeric_features = [
    col for col in [
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_count",
    "is_shared_bath",
    "is_private_bath",
    "is_half_bath",
    "minimum_nights",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "latitude",
    "longitude"
] if col in df.columns
]

categorical_features = [
    col for col in [
        "neighbourhood_cleansed",
        "property_type",
        "room_type",
        "instant_bookable"
    ] if col in df.columns
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['price', 'accommodates', 'bedrooms', 'beds', 'bathrooms_count', 'is_shared_bath', 'is_private_bath', 'is_half_bath', 'minimum_nights', 'availability_365', 'number_of_reviews', 'review_scores_rating', 'review_scores_cleanliness', 'review_scores_location', 'latitude', 'longitude']
Categorical features: ['neighbourhood_cleansed', 'property_type', 'room_type', 'instant_bookable']


Build preprocessing pipelines

In [44]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

Transform structured features

In [46]:
structured_features = preprocessor.fit_transform(df)
structured_features.shape

(4397, 79)

In [49]:
df.isnull().sum().sort_values(ascending=False).head(20)

,0
review_scores_location,962
review_scores_rating,960
review_scores_cleanliness,960
price,913
beds,869
bedrooms,305
description,47
longitude,0
neighbourhood_cleansed,0
name,0


TF-IDF for amenities and description

In [51]:
amenities_vectorizer = TfidfVectorizer(max_features=300, ngram_range=(1, 2))
amenities_features = amenities_vectorizer.fit_transform(df["amenities_clean"].fillna(""))

description_vectorizer = TfidfVectorizer(max_features=300, stop_words="english", ngram_range=(1, 2))
description_features = description_vectorizer.fit_transform(df["description_clean"].fillna(""))

print("Amenities matrix:", amenities_features.shape)
print("Description matrix:", description_features.shape)

Amenities matrix: (4397, 300)
Description matrix: (4397, 300)


Combine all features into final matrix

In [52]:
feature_matrix = hstack([
    csr_matrix(structured_features),
    amenities_features,
    description_features
]).tocsr()

feature_matrix.shape

(4397, 679)

## Build KNN similarity model

In [53]:
nn_model = NearestNeighbors(metric="cosine", algorithm="brute")
nn_model.fit(feature_matrix)

id_to_index = {listing_id: idx for idx, listing_id in enumerate(df["id"])}
index_to_id = {idx: listing_id for listing_id, idx in id_to_index.items()}

Similar listing recommender function

In [54]:
def format_output(result_df):
    preferred_cols = [
        "id",
        "name",
        "neighbourhood_cleansed",
        "property_type",
        "room_type",
        "price",
        "accommodates",
        "bedrooms",
        "beds",
        "review_scores_rating",
        "number_of_reviews",
        "instant_bookable",
        "similarity_score",
        "final_score",
        "amenity_match_count"
    ]
    cols = [col for col in preferred_cols if col in result_df.columns]
    return result_df[cols].reset_index(drop=True)


def recommend_similar_listings(listing_id, top_n=10):
    if listing_id not in id_to_index:
        raise ValueError(f"Listing ID {listing_id} not found.")

    idx = id_to_index[listing_id]
    distances, indices = nn_model.kneighbors(feature_matrix[idx], n_neighbors=top_n + 1)

    rec_indices = indices.flatten()[1:]
    rec_distances = distances.flatten()[1:]

    result = df.iloc[rec_indices].copy()
    result["similarity_score"] = 1 - rec_distances

    return format_output(result)

Test similar listing recommender

In [55]:
sample_listing_id = int(df.iloc[0]["id"])
print("Sample listing ID:", sample_listing_id)

recommend_similar_listings(sample_listing_id, top_n=10)

Sample listing ID: 3781


,id,name,neighbourhood_cleansed,property_type,room_type,price,accommodates,bedrooms,beds,review_scores_rating,number_of_reviews,instant_bookable,similarity_score
0,1201423221609705644,Convenient 1BR close to Airport,East Boston,Entire rental unit,Entire home/apt,131.0,2,1.0,1.0,NaN,0,f,0.874824
1,39788966,"Apartment w/Parking, minutes to airport / Down...",East Boston,Entire rental unit,Entire home/apt,270.0,6,3.0,4.0,4.97,223,f,0.865229
2,1175980636500386344,Stylish 1 bedroom condo in Boston near waterfront,East Boston,Entire rental unit,Entire home/apt,NaN,2,1.0,NaN,NaN,0,f,0.859259
3,776971614888195449,One T stop from Downtown,East Boston,Entire rental unit,Entire home/apt,180.0,2,1.0,2.0,5.00,10,f,0.859065
4,660750253855523277,Modern Apt Central to Logan Airport & Downtown,East Boston,Entire rental unit,Entire home/apt,230.0,5,2.0,4.0,4.97,169,f,0.849844
5,1339122322732605643,Luxury 1BR Next to the Airport 2336,East Boston,Entire rental unit,Entire home/apt,NaN,2,1.0,NaN,4.87,15,f,0.846138
6,1092092772949307191,Cozy East Boston Apt with Patio,East Boston,Entire rental unit,Entire home/apt,179.0,4,3.0,3.0,NaN,0,f,0.845843
7,1350071389732690605,Luxury 1BR Next to Airport 9775,East Boston,Entire rental unit,Entire home/apt,NaN,2,1.0,NaN,4.93,14,f,0.845754
8,1344321556066592352,Luxury 1BR Next to Logan Airport 2491,East Boston,Entire rental unit,Entire home/apt,NaN,2,1.0,NaN,4.80,20,f,0.845081
9,972577050709299536,Cozy Oasis Near Downtown & Airport,East Boston,Entire rental unit,Entire home/apt,NaN,3,NaN,NaN,5.00,2,f,0.843740


In [56]:
df[df['id'] == 3781]

,id,name,description,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,availability_365,number_of_reviews,review_scores_rating,review_scores_cleanliness,review_scores_location,instant_bookable,amenities_clean,description_clean,bathrooms_text_clean,bathrooms_count,is_shared_bath,is_private_bath,is_half_bath
0,3781,HARBORSIDE-Walk to subway,Fully separate apartment in a two apartment bu...,East Boston,42.36413,-71.02991,Entire rental unit,Entire home/apt,2,1 bath,1.0,1.0,"[""Heating"", ""Dishwasher"", ""Smoke alarm"", ""Iron...",125.0,29,326,26,4.96,4.96,4.85,f,heating dishwasher smoke_alarm iron carbon_mon...,fully separate apartment in a two apartment bu...,1 bath,1.0,0,0,0


User preference class

In [57]:
@dataclass
class UserPreferences:
    neighborhood: Optional[str] = None
    room_type: Optional[str] = None
    property_type: Optional[str] = None
    max_price: Optional[float] = None
    min_rating: Optional[float] = None
    accommodates: Optional[int] = None
    min_bedrooms: Optional[float] = None
    instant_bookable: Optional[str] = None
    amenity_keywords: Optional[List[str]] = None

Build synthetic user profile

In [58]:
def build_preference_profile(preferences: UserPreferences):
    profile = {}

    for col in numeric_features:
        profile[col] = df[col].median() if col in df.columns else np.nan

    for col in categorical_features:
        if col in df.columns and not df[col].mode().empty:
            profile[col] = df[col].mode()[0]
        else:
            profile[col] = "Unknown"

    if preferences.max_price is not None and "price" in profile:
        profile["price"] = preferences.max_price
    if preferences.accommodates is not None and "accommodates" in profile:
        profile["accommodates"] = preferences.accommodates
    if preferences.min_bedrooms is not None and "bedrooms" in profile:
        profile["bedrooms"] = preferences.min_bedrooms
    if preferences.min_rating is not None and "review_scores_rating" in profile:
        profile["review_scores_rating"] = preferences.min_rating

    if preferences.neighborhood is not None and "neighbourhood_cleansed" in profile:
        profile["neighbourhood_cleansed"] = preferences.neighborhood
    if preferences.room_type is not None and "room_type" in profile:
        profile["room_type"] = preferences.room_type
    if preferences.property_type is not None and "property_type" in profile:
        profile["property_type"] = preferences.property_type
    if preferences.instant_bookable is not None and "instant_bookable" in profile:
        profile["instant_bookable"] = preferences.instant_bookable

    amenities_text = ""
    if preferences.amenity_keywords:
        amenities_text = " ".join([x.lower().replace(" ", "_") for x in preferences.amenity_keywords])

    profile["amenities_clean"] = amenities_text
    profile["description_clean"] = amenities_text

    return pd.DataFrame([profile])


def transform_profile(profile_df):
    structured = preprocessor.transform(profile_df)
    amenities = amenities_vectorizer.transform(profile_df["amenities_clean"].fillna(""))
    description = description_vectorizer.transform(profile_df["description_clean"].fillna(""))

    return hstack([csr_matrix(structured), amenities, description]).tocsr()

Preference-based personalized recommender

In [59]:
def recommend_from_preferences(preferences: UserPreferences, top_n=10):
    candidate_df = df.copy()

    if preferences.max_price is not None and "price" in candidate_df.columns:
        candidate_df = candidate_df[candidate_df["price"].fillna(np.inf) <= preferences.max_price]

    if preferences.min_rating is not None and "review_scores_rating" in candidate_df.columns:
        candidate_df = candidate_df[candidate_df["review_scores_rating"].fillna(0) >= preferences.min_rating]

    if preferences.neighborhood is not None and "neighbourhood_cleansed" in candidate_df.columns:
        candidate_df = candidate_df[
            candidate_df["neighbourhood_cleansed"].str.lower() == preferences.neighborhood.lower()
        ]

    if preferences.room_type is not None and "room_type" in candidate_df.columns:
        candidate_df = candidate_df[
            candidate_df["room_type"].str.lower() == preferences.room_type.lower()
        ]

    if preferences.property_type is not None and "property_type" in candidate_df.columns:
        candidate_df = candidate_df[
            candidate_df["property_type"].str.lower() == preferences.property_type.lower()
        ]

    if preferences.accommodates is not None and "accommodates" in candidate_df.columns:
        candidate_df = candidate_df[candidate_df["accommodates"].fillna(0) >= preferences.accommodates]

    if preferences.min_bedrooms is not None and "bedrooms" in candidate_df.columns:
        candidate_df = candidate_df[candidate_df["bedrooms"].fillna(0) >= preferences.min_bedrooms]

    if preferences.instant_bookable is not None and "instant_bookable" in candidate_df.columns:
        candidate_df = candidate_df[
            candidate_df["instant_bookable"].str.lower() == preferences.instant_bookable.lower()
        ]

    if candidate_df.empty:
        return pd.DataFrame({"message": ["No listings matched the preference filters."]})

    profile_df = build_preference_profile(preferences)
    profile_features = transform_profile(profile_df)

    candidate_indices = candidate_df.index.tolist()
    candidate_matrix = feature_matrix[candidate_indices]

    similarity_scores = cosine_similarity(profile_features, candidate_matrix).flatten()

    candidate_df = candidate_df.copy()
    candidate_df["similarity_score"] = similarity_scores

    if preferences.amenity_keywords:
        amenity_keywords = [kw.lower().replace(" ", "_") for kw in preferences.amenity_keywords]
        candidate_df["amenity_match_count"] = candidate_df["amenities_clean"].apply(
            lambda x: sum(1 for kw in amenity_keywords if kw in str(x))
        )
        candidate_df["final_score"] = candidate_df["similarity_score"] + 0.05 * candidate_df["amenity_match_count"]
    else:
        candidate_df["amenity_match_count"] = 0
        candidate_df["final_score"] = candidate_df["similarity_score"]

    sort_cols = ["final_score"]
    if "review_scores_rating" in candidate_df.columns:
        sort_cols.append("review_scores_rating")
    if "number_of_reviews" in candidate_df.columns:
        sort_cols.append("number_of_reviews")

    candidate_df = candidate_df.sort_values(by=sort_cols, ascending=False).head(top_n)

    return format_output(candidate_df)

Check available Boston neighborhoods

In [60]:
sorted(df["neighbourhood_cleansed"].dropna().unique().tolist())[:50]

['Allston',
 'Back Bay',
 'Bay Village',
 'Beacon Hill',
 'Brighton',
 'Charlestown',
 'Chinatown',
 'Dorchester',
 'Downtown',
 'East Boston',
 'Fenway',
 'Hyde Park',
 'Jamaica Plain',
 'Leather District',
 'Longwood Medical Area',
 'Mattapan',
 'Mission Hill',
 'North End',
 'Roslindale',
 'Roxbury',
 'South Boston',
 'South Boston Waterfront',
 'South End',
 'West End',
 'West Roxbury']

Run personalized recommendation example

In [61]:
preferences = UserPreferences(
    neighborhood="Back Bay",
    room_type="Entire home/apt",
    max_price=250,
    min_rating=4.5,
    accommodates=2,
    min_bedrooms=1,
    instant_bookable="t",
    amenity_keywords=["wifi", "kitchen", "washer"]
)

recommendations = recommend_from_preferences(preferences, top_n=10)
recommendations

,id,name,neighbourhood_cleansed,property_type,room_type,price,accommodates,bedrooms,beds,review_scores_rating,number_of_reviews,instant_bookable,similarity_score,final_score,amenity_match_count
0,5581575,Private Entrance * King Beds * Parking * Kid R...,Back Bay,Entire rental unit,Entire home/apt,250.0,4,2.0,2.0,4.93,215,t,0.818363,0.968363,3
1,1421363918459951410,"(3-42) Charming 1Bed, Back Bay, Newbury, Fenway!",Back Bay,Entire rental unit,Entire home/apt,232.0,2,1.0,1.0,5.00,9,t,0.803748,0.953748,3
2,981123327835096425,Centrally located 1bed | Newbury Street,Back Bay,Entire rental unit,Entire home/apt,202.0,2,1.0,1.0,4.73,67,t,0.803108,0.953108,3
3,1091227855426817520,"(J10) Back Bay 2 Bedrooms/ 2 TVs, 2 Desks! SMALL",Back Bay,Entire rental unit,Entire home/apt,231.0,3,2.0,2.0,4.54,92,t,0.802661,0.952661,3
4,1420387124346967931,"(3-52) Newly furnish, Back Bay!",Back Bay,Entire rental unit,Entire home/apt,208.0,2,1.0,1.0,4.71,17,t,0.801667,0.951667,3


## Evaluation helper

In [62]:
def evaluate_recommendation_alignment(recommendations, preferences):
    if recommendations.empty or "message" in recommendations.columns:
        return {"status": "No recommendations to evaluate"}

    metrics = {}

    if preferences.max_price is not None and "price" in recommendations.columns:
        metrics["pct_within_budget"] = round((recommendations["price"] <= preferences.max_price).mean() * 100, 2)

    if preferences.room_type is not None and "room_type" in recommendations.columns:
        metrics["pct_room_type_match"] = round(
            (recommendations["room_type"].str.lower() == preferences.room_type.lower()).mean() * 100, 2
        )

    if preferences.neighborhood is not None and "neighbourhood_cleansed" in recommendations.columns:
        metrics["pct_neighborhood_match"] = round(
            (recommendations["neighbourhood_cleansed"].str.lower() == preferences.neighborhood.lower()).mean() * 100, 2
        )

    if "review_scores_rating" in recommendations.columns:
        metrics["avg_recommended_rating"] = round(recommendations["review_scores_rating"].mean(), 2)

    if "property_type" in recommendations.columns:
        metrics["property_type_diversity"] = int(recommendations["property_type"].nunique())

    return metrics

Evaluate personalized recommendations

In [63]:
evaluate_recommendation_alignment(recommendations, preferences)

{'pct_within_budget': np.float64(100.0),
 'pct_room_type_match': np.float64(100.0),
 'pct_neighborhood_match': np.float64(100.0),
 'avg_recommended_rating': np.float64(4.78),
 'property_type_diversity': 1}

## Save outputs

In [64]:
recommendations.to_csv("boston_preference_recommendations.csv", index=False)

similar_listings = recommend_similar_listings(sample_listing_id, top_n=10)
similar_listings.to_csv("boston_similar_listings.csv", index=False)

print("Files saved successfully.")

Files saved successfully.
